Ұяшық 6.1. Test жиынындағы ресми метрикалар

In [1]:
# Ұяшық 6.1 — test жиынындағы түпкілікті метрикалар
import json, time
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path(r"D:\road_damage_yolo")
BEST = PROJECT_ROOT / "runs" / "main_yolo26n_640_e30_s42" / "weights" / "best.pt"
DATA_YAML = PROJECT_ROOT / "configs" / "data.yaml"
TAB_DIR = PROJECT_ROOT / "results" / "tables"
THR = json.loads((PROJECT_ROOT / "configs" / "threshold.json").read_text(encoding="utf-8"))["conf"]
print("val жиынында таңдалған conf:", THR)

model = YOLO(str(BEST))
t0 = time.time()
m = model.val(
    data=str(DATA_YAML), split="test", imgsz=640, batch=16,
    device=0, workers=2, plots=True,
    project=str(PROJECT_ROOT / "runs"), name="test_main_yolo26n_640_e30_s42", exist_ok=True,
)
print(f"Бағалау уақыты: {time.time() - t0:.1f} с")

box = m.box
px = np.asarray(box.px)
Pc = np.asarray(box.p_curve)        # (класс, 1000)
Rc = np.asarray(box.r_curve)
i = int(np.abs(px - THR).argmin())  # conf = THR-ға ең жақын нүкте
print(f"Қисықтағы нүкте: px[{i}] = {px[i]:.3f}")

def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

rows = [{
    "class": "all",
    "P@thr": Pc.mean(0)[i], "R@thr": Rc.mean(0)[i],
    "F1@thr": f1(Pc.mean(0)[i], Rc.mean(0)[i]),
    "mAP50": box.map50, "mAP50-95": box.map,
    "P_ultra": box.mp, "R_ultra": box.mr,
}]
for k, c in enumerate(box.ap_class_index):
    p_u, r_u, ap50, ap = box.class_result(k)
    rows.append({
        "class": m.names[int(c)],
        "P@thr": Pc[k, i], "R@thr": Rc[k, i], "F1@thr": f1(Pc[k, i], Rc[k, i]),
        "mAP50": ap50, "mAP50-95": ap,
        "P_ultra": p_u, "R_ultra": r_u,
    })

tab = pd.DataFrame(rows)
print("\n@thr — val жиынында таңдалған conf шегінде; *_ultra — Ultralytics (тек анықтама)")
print(tab.round(3).to_string(index=False))

out = TAB_DIR / "test_main_yolo26n_640_e30_s42_metrics.csv"
tab.to_csv(out, index=False)
print("\nСақталды:", out)

val жиынында таңдалған conf: 0.239
Ultralytics 8.4.191  Python-3.12.10 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
YOLO26n summary (fused): 121 layers, 2,390,464 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 260.4±26.1 MB/s, size: 84.8 KB)
val: Scanning D:\road_damage_yolo\data\yolo_dataset\labels\test... 856 images, 78 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 856/856 2.4Kit/s 0.4s0.1ss
val: New cache created: D:\road_damage_yolo\data\yolo_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 4.3it/s 12.6s0.2s
                   all        856       1622      0.612      0.531      0.572      0.278
                   D00        270        387      0.611      0.473      0.523      0.241
                   D10        214        398       0.57      0.482      0.512      0.216
                   D20        465        598      0.722      0.649     

Ұяшық 6.2. Диагностика: шатасу матрицасы қай шекпен есептеледі?

In [2]:
# Ұяшық 6.2 — ConfusionMatrix шектерін бастапқы кодтан тексеру
import inspect
from ultralytics.utils import metrics as M

print(inspect.getsource(M.ConfusionMatrix.__init__))
src = inspect.getsource(M.ConfusionMatrix.process_batch)
print("\n".join(l for l in src.splitlines() if "conf" in l or "iou" in l))

    def __init__(self, names: dict[int, str] | None = None, task: str = "detect", save_matches: bool = False):
        """Initialize a ConfusionMatrix instance.

        Args:
            names (dict[int, str], optional): Names of classes, used as labels on the plot.
            task (str, optional): Type of task, one of 'detect', 'semantic', 'classify', or 'obb'.
            save_matches (bool, optional): Save the indices of GTs, TPs, FPs, FNs for visualization.
        """
        names = names if names is not None else {}
        self.task = task
        self.nc = len(names)  # number of classes
        self.matrix = (
            np.zeros((self.nc, self.nc))
            if self.task in {"classify", "semantic"}
            else np.zeros((self.nc + 1, self.nc + 1))
        )
        self.names = names  # name of classes
        self.matches = {} if save_matches else None

        conf: float = 0.25,
        iou_thres: float = 0.45,
        """Update confusion matrix for object detect

Ұяшық 6.3. Жылдамдықты дұрыс өлшеу (бенчмарк)

In [3]:
# Ұяшық 6.3 — Бір суретті өңдеу уақыты және FPS (batch=1, FP32)
import json, time, subprocess
import numpy as np
import pandas as pd
import cv2, torch
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path(r"D:\road_damage_yolo")
BEST = PROJECT_ROOT / "runs" / "main_yolo26n_640_e30_s42" / "weights" / "best.pt"
DS = PROJECT_ROOT / "data" / "yolo_dataset"
TAB_DIR = PROJECT_ROOT / "results" / "tables"
THR = json.loads((PROJECT_ROOT / "configs" / "threshold.json").read_text(encoding="utf-8"))["conf"]
N_IMG, N_WARM, N_ROUNDS = 200, 20, 3

def gpu_state():
    q = subprocess.run(
        ["nvidia-smi", "--query-gpu=temperature.gpu,clocks.sm,power.draw,pstate",
         "--format=csv,noheader"], capture_output=True, text=True)
    return q.stdout.strip()

paths = sorted((DS / "images" / "test").glob("*.jpg"))[:N_IMG]
imgs = [cv2.imread(str(p)) for p in paths]                # жадқа алдын ала жүктеу (BGR)
assert all(im is not None for im in imgs), "Кейбір сурет оқылмады"
print(f"{len(imgs)} сурет жүктелді, өлшемі: {imgs[0].shape}")

model = YOLO(str(BEST))
kw = dict(imgsz=640, conf=THR, device=0, half=False, verbose=False)

print("GPU (басында):", gpu_state())
for im in imgs[:N_WARM]:                                    # қыздыру
    model.predict(im, **kw)
torch.cuda.synchronize()

rec = []
for rnd in range(1, N_ROUNDS + 1):
    for im in imgs:
        t0 = time.perf_counter()
        r = model.predict(im, **kw)[0]
        torch.cuda.synchronize()
        rec.append({"round": rnd, "total_ms": (time.perf_counter() - t0) * 1000,
                    **{f"{k}_ms": v for k, v in r.speed.items()}})
    print(f"GPU ({rnd}-раундтан кейін):", gpu_state())

df = pd.DataFrame(rec)
df.to_csv(TAB_DIR / "benchmark_yolo26n_640_bs1_fp32_raw.csv", index=False)

def summ(g):
    return pd.Series({
        "total_median": g["total_ms"].median(),
        "total_p95": g["total_ms"].quantile(0.95),
        "pre_median": g["preprocess_ms"].median(),
        "inf_median": g["inference_ms"].median(),
        "post_median": g["postprocess_ms"].median(),
        "FPS (1000/total_median)": 1000 / g["total_ms"].median(),
    })

per_round = df.groupby("round").apply(summ)
overall = summ(df).to_frame("all").T
res = pd.concat([per_round, overall])
print("\nУақыт (мс), batch=1, FP32, imgsz=640:")
print(res.round(2).to_string())
res.to_csv(TAB_DIR / "benchmark_yolo26n_640_bs1_fp32_summary.csv")

200 сурет жүктелді, өлшемі: (600, 600, 3)
GPU (басында): 44.50, 300 MHz, 3.74 W, P8
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING 'half' is deprecated and wil

Ұяшық 6.4. Диагностика: валидатор матрицаға қандай conf береді?

In [4]:
# Ұяшық 6.4 — Валидатор шатасу матрицасына қандай conf береді?
import inspect
from ultralytics.models.yolo.detect.val import DetectionValidator

src = inspect.getsource(DetectionValidator)
for n, line in enumerate(src.splitlines(), 1):
    if "confusion_matrix" in line or "args.conf" in line or "conf=" in line:
        print(f"{n:4d}: {line.rstrip()}")

 131:         self.confusion_matrix = ConfusionMatrix(names=model.names, save_matches=self.args.plots and self.args.visualize)
 149:             self.args.conf,
 244:                 self.confusion_matrix.process_batch(predn, pbatch, conf=self.args.conf)
 246:                     self.confusion_matrix.plot_matches(
 271:                 self.confusion_matrix.plot(save_dir=self.save_dir, normalize=normalize, on_plot=self.on_plot)
 273:         self.metrics.confusion_matrix = self.confusion_matrix
 317:             matrix = torch.as_tensor(self.confusion_matrix.matrix, device=self.device)
 320:                 self.confusion_matrix.matrix = matrix.cpu().numpy()
 472:         ).save_txt(file, save_conf=save_conf)
